# API Documentation Agent

This notebook walks through the full pipeline: it downloads and chunks an OpenAPI specification, indexes those chunks for retrieval, and launches an interactive Gradio chat interface where you can ask natural-language questions about the API and receive answers grounded in the official documentation.

By default, retrieval runs on a local Chroma index (no Vertex AI Search data store needed) — set `SEARCH_BACKEND` to `vertex` in the config cell below if you want to index into Vertex AI Search instead. Gemini generation always runs on Vertex AI either way, so a GCP project is required.

The Kubernetes API is used as the default example, but any OpenAPI 2.0/3.0 spec, Postman collection, or Markdown documentation works.

In [ ]:
!pip install -q uv
!uv pip install -q --system gradio google-api-core google-cloud-discoveryengine google-cloud-aiplatform==1.71.1 vertexai requests pyyaml 'mcp[cli]' chromadb

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
GCP_PROJECT_ID = "your-project-id"                # @param {type:"string"}
GCP_LOCATION = "global"                            # @param {type:"string"}
GEMINI_LOCATION = "us-central1"                   # @param {type:"string"}
SEARCH_BACKEND = "local"                          # @param ["local", "vertex"]
VERTEX_SEARCH_DATA_STORE_ID = "your-engine-id"    # @param {type:"string"} (only used when SEARCH_BACKEND is "vertex")
API_SPEC_URL = "https://raw.githubusercontent.com/kubernetes/kubernetes/v1.36.0/api/openapi-spec/swagger.json"  # @param {type:"string"}
API_NAME = "Kubernetes"                            # @param {type:"string"}
API_VERSION = "v1.36.0"                            # @param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["GCP_LOCATION"] = GCP_LOCATION
os.environ["GEMINI_LOCATION"] = GEMINI_LOCATION
os.environ["SEARCH_BACKEND"] = SEARCH_BACKEND
os.environ["API_NAME"] = API_NAME

if SEARCH_BACKEND == "vertex":
    os.environ["VERTEX_SEARCH_DATA_STORE_ID"] = VERTEX_SEARCH_DATA_STORE_ID
else:
    os.environ["LOCAL_COLLECTION"] = API_NAME.lower()

## Step 1: Ingest and index API documentation

Parses the OpenAPI spec, writes one chunk per endpoint and one per schema to `data/<name>_<version>_chunks.jsonl`, and indexes them into the backend selected above (`--upload`).

In [ ]:
!python -m src.ingest --spec {API_SPEC_URL} --name {API_NAME.lower()} --version {API_VERSION} --upload --backend {SEARCH_BACKEND}

## Step 2: (Vertex AI Search backend only) Verify the data store

If you set `SEARCH_BACKEND = "vertex"` above, the ingest step already created the data store and imported your chunks — no console steps required. Optionally:

1. Check the [Vertex AI Search console](https://console.cloud.google.com/gen-app-builder/data-stores) to confirm indexing finished.
2. If you're attaching this data store to an existing **Engine**, go to [Engines](https://console.cloud.google.com/gen-app-builder/engines), create or reuse one pointing at the data store, and put the **Engine ID** into `VERTEX_SEARCH_DATA_STORE_ID` in the config cell above (then re-run that cell).

If you're using the default local backend, skip this step — nothing more to do.

## Step 3: Launch the Documentation Agent

Colab has no `localhost` to open, so this always uses a public Gradio share link — but it's protected with a randomly generated login printed below so the link isn't wide open to anyone who finds it.

In [ ]:
import os
import secrets
os.chdir("/content/api-rag")  # adjust if repo is cloned elsewhere

from src.app import demo

username = "colab"
password = secrets.token_urlsafe(12)
print(f"Login for the shared link -> username: {username}  password: {password}")

demo.launch(share=True, auth=(username, password))